#**Sector-Specific Government Policy Tracker**

OVERVIEW: This project demonstrates a basic ETL pipeline and analysis of US agency policy documents that may affect the tech industry. Sequentially, it:
1. Pulls all Federal Register documents instances within the past 6 months
2. Basic cleaning/munging
3. Fetching the actual documents themselves.
4. Transformer based model comparison to a given topic, industry, sector, etc.

Step 1. Data Pull

In [ ]:
import requests
import pandas as pd
import datetime
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
import time
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm

In [ ]:
#Preemptively
pd.set_option('display.max_columns', None)

In [ ]:
#Calculate the date for six months ago
six_months_ago = (datetime.date.today() - datetime.timedelta(days=180)).isoformat()

#define API endpoint - This is quite long because the basic request doesn't pull all fields, and for exploratory reasons I'd like to take a look at them all.

#Here, we filter out notices, since these are usually junk
url = 'https://www.federalregister.gov/api/v1/documents.json?fields[]=abstract&fields[]=action&fields[]=agencies&fields[]=agency_names&fields[]=cfr_topics&fields[]=citation&fields[]=comments_close_on&fields[]=correction_of&fields[]=dates&fields[]=document_number&fields[]=effective_on&fields[]=end_page&fields[]=excerpts&fields[]=executive_order_notes&fields[]=executive_order_number&fields[]=explanation&fields[]=full_text_xml_url&fields[]=president&fields[]=presidential_document_number&fields[]=proclamation_number&fields[]=public_inspection_pdf_url&fields[]=publication_date&fields[]=raw_text_url&fields[]=significant&fields[]=signing_date&fields[]=start_page&fields[]=subtype&fields[]=title&fields[]=toc_doc&fields[]=toc_subject&fields[]=topics&fields[]=type&conditions[type][]=RULE&conditions[type][]=PRORULE&conditions[type][]=PRESDOCU&fields[]=volume&per_page='

params = {
    'per_page': 1000,
    'conditions[publication_date][gte]': six_months_ago
}

#Fetch
response = requests.get(url, params=params)
response.raise_for_status()
data = response.json()

data['results'][0] #Let's take a look at a response

Now let's flatten the data for readability's sake

In [ ]:
#Flatten
df = pd.json_normalize(data['results'])

print(f"Retrieved {len(df)} documents.")
display(df.head())

It appears that the endpoint doesn't immediately expose the entire text of a document itself, which we'll likely need for the best cosine similarity match later.

Let's grab that before we begin. it appears it can be fetched from the raw_text_url endpoint.

It appears that the columns ***document number*** and ***date*** can be used to extract individual full text documents. Let's write a function to do this.
- However, per the endpoint, publication_date must be in YYYY/MM/DD format. Let's create that now.

In [ ]:
df['publication_date'] = pd.to_datetime(df['publication_date']).dt.strftime('%Y/%m/%d')
df['publication_date'][0]

In [ ]:
#Function to retrieve full text
def get_full_text(df, doc_num, date):
  full_text_url = f'https://www.federalregister.gov/documents/full_text/xml/{date}/{doc_num}.xml'
  response = requests.get(full_text_url)
  return response.text

#Test
get_full_text(df, df['document_number'][0], df['publication_date'][0])

**Step 2 - Cleaning and Munging**

Now that our data is tabular - we can actually begin to understand it and evaluate it for cleaning.

**NOTE**: For the explicit purposes of this project, the only inputs we will ultimately need are two lists:
- A list/desc of sectors
- A list of text to match against via our bi-encoder model.

However, for the sake of this exercise, let's do some basic cleaning anyway - who knows, we might need it later.

In [ ]:
#Check for irrelevant/less useful columns
df_cols_check = df.isna().sum()

It looks like our data has a lot of empty cols - let's retain the raw data, but make a copy that's easier to work with that excludes cols > 99% null.

- **NOTE**: It should be noted that contextually, sometimes we would want to keep columns like this. However, for this specific purpose, we're fine to remove.

In [ ]:
df_cols_check = df.isna().sum() #Check empties
df_cols_GT_99_percent_empty = df_cols_check[df_cols_check > df.shape[0] * .99] #Flag emptys > 99%

df_cols_to_remove = df_cols_GT_99_percent_empty.index.to_list() #These are the cols we'll remove

print(df_cols_to_remove)

In [ ]:
df2 = df.drop(columns=df_cols_to_remove).copy() #Drop the cols
df2.head()

Documents of type "Notice" aren't important for our purposes - these include lesser agency-specific things, such as meeting announcements. This would likely be noise in eventual results for our tracker.

In [ ]:
df2 = df2[~df2['type'].str.contains('Notice', case=False, na=False)]
df2['type'].value_counts() #Just making sure



Let's visualize publications over time and by agency, just to give ourselves a bit of context.

In [ ]:
#Ensure publication_date is datetime for proper sorting
df_plot = df2.copy()
df_plot['publication_date'] = pd.to_datetime(df_plot['publication_date'])

#Group by date and count publications
daily_counts = df_plot.groupby('publication_date').size()

#Plotting
plt.figure(figsize=(6, 3))
daily_counts.plot(kind='line', marker='o', color='#1a73e8', linewidth=2)
plt.title('Federal Register Publications Over Time (Filtered)', fontsize=14)
plt.xlabel('Publication Date', fontsize=12)
plt.ylabel('Number of Documents', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
#Explode the agency_names list to count individual agency occurrences
agency_counts = df2['agency_names'].explode().value_counts().head(10)

#Plotting
plt.figure(figsize=(10, 6))
agency_counts.plot(kind='barh', color='#34a853')
plt.title('Top 10 Most Active Agencies (Filtered)', fontsize=14)
plt.xlabel('Number of Documents', fontsize=12)
plt.ylabel('Agency Name', fontsize=12)
plt.gca().invert_yaxis()  #Put the highest count at the top
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
#Let's see what a single document looks like
def fetch_full_text_doc(url):
  """
  Fetches singular FR doc, returns as xml, and parses Federal Register XML and concatenates text from all <FP> tags: FP indicates the actual text
  """
  response = requests.get(url)
  doc_xml =  response.text
  soup = BeautifulSoup(doc_xml, 'xml')
  fp_tags = soup.find_all('FP')
  # Extract text from each tag and join with spaces
  clean_text = " ".join([tag.get_text(strip=True) for tag in fp_tags])
  return clean_text

sample = fetch_full_text_doc('https://www.federalregister.gov/documents/full_text/xml/2026/06/18/2026-12435.xml')

sample

It works - let's have it go through the whole dataframe.
- **NOTE**: Although normally using .map() would be quicker, in this case we're just going to make a loop to avoid any throttling risk.

In [91]:
# We will use a list to store the results then add to the dataframe
full_texts = []

print(f"Starting batch retrieval for {len(df2)} documents...")

# Loop over the filtered dataframe
for index, row in tqdm(df2.iterrows(), total=df2.shape[0]):
    url = row['full_text_xml_url']
    if pd.isna(url):
        full_texts.append("")
        continue

    try:
        # Using the helper function defined in the previous cell
        text = fetch_full_text_doc(url)
        full_texts.append(text)
    except Exception as e:
        print(f"Error fetching index {index}: {e}")
        full_texts.append("")

    # Sleep for 3 seconds to avoid throttling
    time.sleep(0.5)

# Assign the results to a new column
df2['full_text'] = full_texts


print("Done! Full text column added.")
display(df2[['title', 'full_text']].head())

Done! Full text column added.


,title,full_text
0,"Flag Day and National Flag Week, 2026","On June 14, 1777, the delegates of the Second ..."
1,"National Homeownership Month, 2026","During National Homeownership Month, my Admini..."
2,Final Waivers and Extensions of the Project Pe...,
3,Airworthiness Directives; Airbus Helicopters,Airbus Helicopters:Docket No. FAA-2026-4657; P...
4,Request for Information (RFI): Pharmacy Benefi...,++ Affiliated provider group ++ Data vendors +...


Now we have our dataframe, complete enough for our purposes. Let's extract the full text articles for input

In [92]:
industry_profile_semis = """
Semiconductor manufacturing, chip fabrication,
advanced packaging, wafer processing,
photolithography, foundries, integrated circuits,
microelectronics supply chain, semiconductor workforce
"""


industry_profile_ai = """Organizations developing, deploying, or using artificial intelligence technologies, including machine learning, generative AI, large language models, computer vision, natural language processing, AI infrastructure, model training, AI safety, and responsible AI governance."""


industry_profile_biotech = """This cluster encompasses federal oversight of healthcare delivery, medical innovation, biomedical research, and reimbursement systems. Regulatory activity spans medical device approvals, pharmaceutical oversight, clinical research governance, Medicare and Medicaid payment structures, health information management, and public-health reporting requirements. The sector sits at the intersection of patient safety, scientific advancement, and healthcare economics, making it one of the most consistently active areas of federal rulemaking. Changes in this domain can affect hospitals, insurers, biotechnology firms, medical-device manufacturers, pharmaceutical companies, research institutions, and healthcare technology vendors."""

industry_profile_cybersecurity = """Cybersecurity regulation increasingly functions as a cross-sector governance framework rather than a standalone technology issue. Federal actions in this space address cyber risk management, incident reporting, software security, supply-chain resilience, operational technology protection, and security standards for critical infrastructure operators. The sector reflects growing concerns that cyber incidents can disrupt essential services such as energy, healthcare, transportation, telecommunications, and financial systems. As a result, cybersecurity requirements are becoming embedded into broader compliance and enterprise-risk management programs across the economy."""

industry_profile_energy = """This theme encompasses the production, transmission, regulation, and security of energy resources and supporting infrastructure. Federal activity frequently addresses power generation, grid reliability, nuclear licensing, infrastructure modernization, operational safety, and resilience planning. Many regulatory initiatives are driven by concerns over energy security, reliability, technological modernization, and long-term capacity needs. Stakeholders include utilities, independent power producers, infrastructure developers, engineering firms, nuclear operators, and large industrial energy consumers."""

industry_profile_maritime_transport = """Transportation-related rulemaking focuses on the safe, efficient, and reliable movement of people and goods. Federal actions can involve vehicle safety standards, maritime operations, logistics systems, transportation infrastructure, regulatory compliance requirements, and operational risk management. The sector plays a foundational role in economic activity because transportation networks connect supply chains, labor markets, and consumer demand. Regulatory changes often seek to improve safety, efficiency, resilience, and interoperability across transportation modes."""

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

query = industry_profile_cybersecurity

query_emb = model.encode([query])

doc_embs = model.encode(
    df2["full_text"].fillna("").tolist(),
    show_progress_bar=True
)

scores = cosine_similarity(query_emb, doc_embs)[0]

df2["similarity"] = scores

results = df2.sort_values("similarity", ascending=False)

# Display the top matches
display(results[["title", "agency_names", "similarity"]].head(10))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]